# Sentiment Analysis Module

In [1]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /home/asish-
[nltk_data]     jose/nltk_data...


True

In [1]:
from nltk.sentiment import SentimentIntensityAnalyzer

class SentimentAnalyzer:
    
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()

    def analyze(self, text):
        scores = self.analyzer.polarity_scores(text)
        compound = scores['compound']

        sentiment_level = self.classify_sentiment(compound)

        return {
            "scores":scores,
            "compound":compound,
            "level": sentiment_level
        }
    
    def classify_sentiment(self, compound):

        if compound >= 0.3:
            return "positive"
        
        elif -0.3 <= compound < 0.3:
            return "neutral"
        
        elif -0.6 <= compound < -0.3:
            return "concerned"
        
        else:
            return "distressed"

In [11]:

analyzer = SentimentAnalyzer()

text = "I am really scared about my cancer diagnosis"

result = analyzer.analyze(text)

print(result)

{'scores': {'neg': 0.603, 'neu': 0.397, 'pos': 0.0, 'compound': -0.8221}, 'compound': -0.8221, 'level': 'distressed'}


In [1]:
def build_prompt(user_query, context, sentiment):

    base_prompt = f"""
    You are a medical report explanation assistant.
    Explain clearly without giving medical advice.

    User query: {user_query}
    Context: {context}
    """

    if sentiment == "distressed":
        tone = "Respond in a very empathetic and supportive tone."
    elif sentiment == "concerned":
        tone = "Respond in a calm and reassuring tone."
    else:
        tone = "Respond in a clear and neutral tone."

    return base_prompt + "\n" + tone

## Sentiment Analyzer Documentation

This section explains how the sentiment analyzer in this notebook is built and how it is intended to be used.

### Objective

The sentiment analyzer is designed to detect the emotional tone of a user's message in a medical-assistant setting. Its purpose is not only to classify sentiment, but also to help the assistant choose a better response tone.

The notebook maps user input into one of four sentiment levels:

- `positive`
- `neutral`
- `concerned`
- `distressed`

### Approach Used

This notebook uses a lexicon-based sentiment analysis approach with NLTK's VADER sentiment engine.

This is not a trained machine learning model. There is no dataset, train-test split, training step, or saved classifier. Instead, the notebook uses VADER's built-in sentiment lexicon and rule-based scoring.

### 1. Load the VADER Lexicon

The notebook first downloads the required VADER resource:

```python
import nltk
nltk.download('vader_lexicon')
```

This provides the sentiment vocabulary and heuristics used by the analyzer.

### 2. Create the Analyzer Class

The notebook defines a custom `SentimentAnalyzer` wrapper around `SentimentIntensityAnalyzer`:

```python
from nltk.sentiment import SentimentIntensityAnalyzer

class SentimentAnalyzer:
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()
```

The VADER analyzer is initialized once in the constructor and reused for predictions.

### 3. Analyze the Input Text

The main logic is implemented in `analyze()`:

```python
def analyze(self, text):
    scores = self.analyzer.polarity_scores(text)
    compound = scores['compound']
    sentiment_level = self.classify_sentiment(compound)

    return {
        "scores": scores,
        "compound": compound,
        "level": sentiment_level
    }
```

For each input sentence, VADER returns:

- `neg`: negative sentiment proportion
- `neu`: neutral sentiment proportion
- `pos`: positive sentiment proportion
- `compound`: normalized overall sentiment score from `-1` to `1`

The notebook uses the `compound` score as the main value for final sentiment classification.

### 4. Map the Compound Score to Custom Sentiment Labels

The notebook defines custom thresholds in `classify_sentiment()`:

```python
def classify_sentiment(self, compound):
    if compound >= 0.3:
        return "positive"
    elif -0.3 <= compound < 0.3:
        return "neutral"
    elif -0.6 <= compound < -0.3:
        return "concerned"
    else:
        return "distressed"
```

Threshold mapping:

- `compound >= 0.3` -> `positive`
- `-0.3 <= compound < 0.3` -> `neutral`
- `-0.6 <= compound < -0.3` -> `concerned`
- `compound < -0.6` -> `distressed`

These custom labels are useful in a medical support setting because they separate moderate concern from stronger emotional distress.

### Example Output

The notebook tests the analyzer with this sentence:

```python
text = "I am really scared about my cancer diagnosis"
result = analyzer.analyze(text)
print(result)
```

Recorded output from the notebook:

```python
{'scores': {'neg': 0.603, 'neu': 0.397, 'pos': 0.0, 'compound': -0.8221}, 'compound': -0.8221, 'level': 'distressed'}
```

This indicates strongly negative emotional language and places the message in the `distressed` category.

### Prompt Adaptation

The notebook also defines a `build_prompt()` helper function that changes the assistant tone based on sentiment:

```python
def build_prompt(user_query, context, sentiment):
    base_prompt = f"""
    You are a medical report explanation assistant.
    Explain clearly without giving medical advice.

    User query: {user_query}
    Context: {context}
    """

    if sentiment == "distressed":
        tone = "Respond in a very empathetic and supportive tone."
    elif sentiment == "concerned":
        tone = "Respond in a calm and reassuring tone."
    else:
        tone = "Respond in a clear and neutral tone."

    return base_prompt + "\n" + tone
```

This creates the intended workflow:

1. Analyze the emotional tone of the user message.
2. Convert the VADER score into a custom sentiment level.
3. Use that level to adjust the wording style of the generated response.

### End-to-End Flow

The implemented pipeline is:

`User Text -> VADER Scores -> Compound Score -> Custom Sentiment Label -> Tone-Aware Prompt`

### Strengths

- simple and fast
- no training data required
- easy to integrate into a conversational assistant
- custom labels are more useful than a basic negative label for support scenarios
- sentiment result can directly control prompt tone

### Current Limitations

- depends on a general-purpose lexicon rather than domain-specific training
- thresholds are manually chosen and not validated on labeled data
- no evaluation metrics are included in the notebook
- limited handling of complex context, sarcasm, or mixed emotions
- no persistence step is needed because there is no trained model artifact

### Possible Improvements

- evaluate the thresholds on labeled healthcare sentiment examples
- add more emotion-oriented test cases such as fear, confusion, relief, and frustration
- compare VADER with a transformer-based sentiment model
- add fallback handling for ambiguous sentiment scores
- make prompt adaptation more granular across all four sentiment levels

### Summary

This sentiment analyzer is a lightweight rule-based module built with NLTK VADER. It computes sentiment scores, converts the compound score into one of four custom emotional labels, and uses that label to influence assistant tone in a medical-support workflow. It is practical for prototyping, but it would benefit from domain-specific evaluation and calibration.